# Seattle Weather Prediction: Classifying the Day's Weather from Four Measurements

## Goal
Predict the weather type of a day (drizzle, fog, rain, snow or sun) from its precipitation, maximum and minimum temperature, and wind speed.

## Data
- 1,461 days of Seattle weather, 1 Jan 2012 to 31 Dec 2015, with no missing values and no duplicate rows
- Five weather types: rain 641 days (43.9%), sun 640 (43.8%), fog 101 (6.9%), drizzle 53 (3.6%), snow 26 (1.8%)
- Rain and sun make up 88% of the days, so the other three types are rare
- Drizzle, fog and sun days all average 0.00 precipitation, so they are hard to tell apart from these measurements. Rain averages 6.56 and snow 8.55.

## Method
1. Turn the weather names into numbers (0 to 4) and use precipitation, maximum temperature, minimum temperature and wind as inputs. The date was not used.
2. Split 80/20 with the mix of weather types kept the same in both sets: 1,168 days to train and 293 to test (drizzle 11, fog 20, rain 129, snow 5, sun 128).
3. Model: XGBoost classifier, compared with 5-fold cross-validation on the training data only.
4. Tuned the learning rate and gamma with grid search (16 combinations, scored by accuracy) and tested giving rare weather types more weight.
5. Judged by accuracy and by macro F1, which averages the score across the five types so the rare ones count as much as rain and sun.

## Model comparison (5-fold cross-validation on the training data)
| Model | Accuracy | Macro F1 |
|---|---|---|
| Always guess the most common type | 0.438 | n/a |
| XGBoost, default settings | 0.823 | 0.533 |
| XGBoost, tuned (gamma 1, learning rate 0.01) | 0.853 | 0.425 |
| XGBoost, default + class weights | 0.793 | 0.541 |

## Final result (tuned model, test set of 293 days)
- **Accuracy: 0.843**, **macro F1: 0.473**
- Rain: precision 0.97, recall 0.91
- Sun: precision 0.75, recall 1.00
- Snow: precision 1.00, recall 0.40 (2 of the 5 snow days)
- Drizzle and fog: nothing found. All 11 drizzle days and all 20 fog days were predicted as sun.

## Conclusions
- The model is 84% accurate on new days, against 44% for always guessing the most common type, and the test score (0.843) is close to the cross-validated one (0.853).
- In practice it works as a wet-day or dry-day detector. It separates rain and snow from dry days well, but it cannot tell drizzle or fog from sun, because measurements such as visibility, cloud cover and humidity are not in the data.
- Tuning raised accuracy but lowered macro F1 (0.533 to 0.425), because the model gave up on the rare types. There is a trade-off between overall accuracy and the rare types.
- Class weights recovered very little (macro F1 0.541 against 0.533) and cost 3 points of accuracy, so they were not used.
- The tuned model was chosen from the cross-validation results before looking at the test set.

## Limitations and next steps
- Only four numeric measurements were used. Adding humidity, cloud cover or visibility would likely help identify fog and drizzle.
- The test set holds only 5 snow, 11 drizzle and 20 fog days, so the scores for these types are noisy.
- Four years of data from one city.
- This is a learning project, not a forecasting tool.

Importing pandas to load and work with the data.

In [1]:
import pandas as pd

Loading the Seattle weather data into a dataframe.

In [2]:
df = pd.read_csv("seattle-weather.csv")

Looking at the first few rows to see what we're working with.

In [3]:
df.head()

,date,precipitation,temp_max,temp_min,wind,weather
0,1/1/2012,0.0,12.8,5.0,4.7,drizzle
1,1/2/2012,10.9,10.6,2.8,4.5,rain
2,1/3/2012,0.8,11.7,7.2,2.3,rain
3,1/4/2012,20.3,12.2,5.6,4.7,rain
4,1/5/2012,1.3,8.9,2.8,6.1,rain


Checking for missing values in each column.

In [4]:
df.isnull().sum()

date             0
precipitation    0
temp_max         0
temp_min         0
wind             0
weather          0
dtype: int64

Encoding the weather column into numbers so the model can use it, then printing the categories it found.

In [5]:
def labelencoding(c):
    from sklearn import preprocessing
    
    le = preprocessing.LabelEncoder()
    df[c] = le.fit_transform(df[c])
    
    print(df[c].unique())

labelencoding("weather")

df

[0 2 4 3 1]


,date,precipitation,temp_max,temp_min,wind,weather
0,1/1/2012,0.0,12.8,5.0,4.7,0
1,1/2/2012,10.9,10.6,2.8,4.5,2
2,1/3/2012,0.8,11.7,7.2,2.3,2
3,1/4/2012,20.3,12.2,5.6,4.7,2
4,1/5/2012,1.3,8.9,2.8,6.1,2
...,...,...,...,...,...,...
1456,12/27/2015,8.6,4.4,1.7,2.9,2
1457,12/28/2015,1.5,5.0,1.7,1.3,2
1458,12/29/2015,0.0,7.2,0.6,2.6,1
1459,12/30/2015,0.0,5.6,-1.0,3.4,4


Listing the columns that need to be scaled.

In [6]:
cols = ['precipitation' , 'temp_max', 'temp_min', 'wind']

Normalizing those columns by dividing by their max value, so they're all between 0 and 1.

In [7]:
def normalize(df,cols):
    for x in cols:
        df[x] = df[x]/df[x].max()
normalize(df,cols)
df

,date,precipitation,temp_max,temp_min,wind,weather
0,1/1/2012,0.000000,0.359551,0.273224,0.494737,0
1,1/2/2012,0.194991,0.297753,0.153005,0.473684,2
2,1/3/2012,0.014311,0.328652,0.393443,0.242105,2
3,1/4/2012,0.363148,0.342697,0.306011,0.494737,2
4,1/5/2012,0.023256,0.250000,0.153005,0.642105,2
...,...,...,...,...,...,...
1456,12/27/2015,0.153846,0.123596,0.092896,0.305263,2
1457,12/28/2015,0.026834,0.140449,0.092896,0.136842,2
1458,12/29/2015,0.000000,0.202247,0.032787,0.273684,1
1459,12/30/2015,0.000000,0.157303,-0.054645,0.357895,4


Dropping the date column since it's not used as a feature.

In [8]:
df = df.drop('date',axis=1)
df

,precipitation,temp_max,temp_min,wind,weather
0,0.000000,0.359551,0.273224,0.494737,0
1,0.194991,0.297753,0.153005,0.473684,2
2,0.014311,0.328652,0.393443,0.242105,2
3,0.363148,0.342697,0.306011,0.494737,2
4,0.023256,0.250000,0.153005,0.642105,2
...,...,...,...,...,...
1456,0.153846,0.123596,0.092896,0.305263,2
1457,0.026834,0.140449,0.092896,0.136842,2
1458,0.000000,0.202247,0.032787,0.273684,1
1459,0.000000,0.157303,-0.054645,0.357895,4


Splitting the data into features (x) and the target we're predicting (y).

In [9]:
x = df.drop('weather',axis=1)
y = df['weather']

Splitting into training and test sets, holding back 20% for testing.

In [10]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=0)

Installing xgboost.

In [11]:
pip install xgboost

Note: you may need to restart the kernel to use updated packages.


Importing the XGBoost classifier.

In [12]:
from xgboost import XGBClassifier

Training an XGBoost model as a first baseline.

In [13]:
xg = XGBClassifier()
xg.fit(x_train, y_train)

,"objective objective: str | xgboost.objective.Objective | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) 

Making predictions on the test set.

In [14]:
y_hat = xg.predict(x_test)
print (y_hat)

[4 0 2 4 1 2 2 2 4 2 4 2 4 4 1 4 4 2 2 2 2 2 2 4 0 0 4 2 4 4 2 4 2 2 4 2 4
 2 4 2 4 2 2 2 4 1 0 4 4 4 4 4 4 2 2 4 4 2 4 2 4 4 4 4 1 2 4 4 4 2 2 2 2 1
 4 4 4 4 2 2 4 4 2 2 2 0 4 2 4 2 4 4 2 4 2 3 4 4 4 4 2 2 4 2 2 4 2 2 4 4 2
 4 2 4 4 4 4 2 2 2 4 2 2 4 2 4 4 4 4 2 4 2 4 2 2 2 4 4 4 4 2 2 4 4 4 2 2 4
 1 4 4 4 4 2 1 2 4 1 2 4 4 0 2 4 2 2 1 2 2 2 4 4 4 4 2 2 4 4 4 4 4 4 2 4 2
 1 4 1 2 4 4 4 4 4 2 2 4 4 4 4 2 4 4 4 2 2 2 4 4 1 2 2 2 2 1 4 4 2 2 2 4 4
 4 2 4 4 4 4 2 4 0 2 2 2 2 4 2 1 2 2 4 2 4 2 2 4 2 4 0 4 4 4 2 3 4 1 4 4 4
 4 2 1 4 4 4 2 4 2 4 1 4 4 2 2 0 4 2 4 2 4 2 2 2 4 0 4 2 4 2 4 4 2 4]


Looking at the actual test labels to compare against.

In [15]:
print(y_test)

530    4
657    4
459    2
279    4
656    4
      ..
440    2
634    4
494    4
61     2
517    4
Name: weather, Length: 293, dtype: int64


Importing the metrics needed to evaluate the model.

In [16]:
from sklearn.metrics import classification_report, accuracy_score

Checking accuracy and the full classification report.

In [17]:
print ("accuracy score = ", accuracy_score(y_test, y_hat))
print ( "Classification Report", classification_report(y_test, y_hat))

accuracy score =  0.757679180887372
Classification Report               precision    recall  f1-score   support

           0       0.00      0.00      0.00        10
           1       0.24      0.14      0.17        29
           2       0.96      0.91      0.93       123
           3       1.00      0.33      0.50         6
           4       0.71      0.83      0.76       125

    accuracy                           0.76       293
   macro avg       0.58      0.44      0.47       293
weighted avg       0.75      0.76      0.75       293



Checking the model's current parameters before tuning.

In [18]:
xg.get_params()

{'objective': 'multi:softprob',
 'base_score': None,
 'booster': None,
 'callbacks': None,
 'colsample_bylevel': None,
 'colsample_bynode': None,
 'colsample_bytree': None,
 'device': None,
 'early_stopping_rounds': None,
 'enable_categorical': True,
 'eval_metric': None,
 'feature_types': None,
 'feature_weights': None,
 'gamma': None,
 'grow_policy': None,
 'importance_type': None,
 'interaction_constraints': None,
 'learning_rate': None,
 'max_bin': None,
 'max_cat_threshold': None,
 'max_cat_to_onehot': None,
 'max_delta_step': None,
 'max_depth': None,
 'max_leaves': None,
 'min_child_weight': None,
 'missing': nan,
 'monotone_constraints': None,
 'multi_strategy': None,
 'n_estimators': None,
 'n_jobs': None,
 'num_parallel_tree': None,
 'random_state': None,
 'reg_alpha': None,
 'reg_lambda': None,
 'sampling_method': None,
 'scale_pos_weight': None,
 'subsample': None,
 'tree_method': None,
 'validate_parameters': None,
 'verbosity': None}

Setting up a grid of learning rate and gamma values to try.

In [19]:
grid = {
    'learning_rate': [0.1, 1, 0.01, 0.001],
    'gamma': [0, 1, 10, 100]
}

Importing GridSearchCV.

In [20]:
from sklearn.model_selection import GridSearchCV

Setting up a grid search over those parameters, using 10 fold cross validation.

In [21]:
model = GridSearchCV(xg, grid, cv = 10, verbose = 2)

Running the grid search. This tries every combination so it can take a while.

In [22]:
model.fit(x_train, y_train)

Fitting 10 folds for each of 16 candidates, totalling 160 fits
[CV] END .........................gamma=0, learning_rate=0.1; total time=   0.7s
[CV] END .........................gamma=0, learning_rate=0.1; total time=   0.7s
[CV] END .........................gamma=0, learning_rate=0.1; total time=   0.7s
[CV] END .........................gamma=0, learning_rate=0.1; total time=   0.7s
[CV] END .........................gamma=0, learning_rate=0.1; total time=   0.6s
[CV] END .........................gamma=0, learning_rate=0.1; total time=   0.7s
[CV] END .........................gamma=0, learning_rate=0.1; total time=   0.6s
[CV] END .........................gamma=0, learning_rate=0.1; total time=   0.7s
[CV] END .........................gamma=0, learning_rate=0.1; total time=   0.7s
[CV] END .........................gamma=0, learning_rate=0.1; total time=   0.7s
[CV] END ...........................gamma=0, learning_rate=1; total time=   0.6s
[CV] END ...........................gamma=0, l

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBClassifier...ree=None, ...)"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'gamma': [0, 1, ...], 'learning_rate': [0.1, 1, ...]}"
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",10
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",2
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to desi

Using the best model the grid search found to make predictions on the test set.

In [23]:
grid_predict = model.predict(x_test)

print(grid_predict)

[4 1 2 4 4 2 2 2 4 2 4 2 4 4 4 4 4 2 2 2 2 2 2 4 4 4 4 2 4 4 2 4 2 2 4 2 4
 2 4 2 4 2 2 2 4 4 0 4 4 4 4 4 4 2 2 4 4 2 4 2 4 4 4 4 4 2 4 4 4 2 2 2 2 4
 4 4 4 4 2 2 4 4 2 2 2 4 4 2 4 2 4 4 2 4 2 2 4 4 4 4 2 4 4 2 2 4 2 2 4 4 2
 4 2 4 4 4 4 4 2 2 4 2 2 4 2 4 4 4 4 2 4 2 4 2 2 2 4 4 4 4 2 2 4 4 4 2 2 4
 4 4 4 4 4 2 4 2 4 4 2 4 4 4 2 4 2 2 4 2 2 2 4 4 4 4 2 2 4 4 4 4 4 4 2 4 2
 4 4 4 2 4 4 4 4 4 2 2 4 4 4 4 2 4 4 4 2 2 2 4 4 4 2 2 2 2 4 4 4 2 2 2 4 4
 4 2 4 4 4 4 2 4 4 2 2 2 2 4 2 4 2 2 4 2 4 2 2 4 2 4 4 4 4 4 2 3 4 4 4 4 4
 4 3 1 4 4 4 2 4 2 4 1 4 4 2 2 4 4 2 4 2 4 2 2 2 4 4 4 2 4 2 4 4 2 4]


Checking accuracy and the classification report for the tuned model.

In [24]:
print ("Accuracy Score = ", accuracy_score(y_test,grid_predict))
print("Classification Report = ",classification_report(y_test,grid_predict))

Accuracy Score =  0.8088737201365188
Classification Report =                precision    recall  f1-score   support

           0       0.00      0.00      0.00        10
           1       0.33      0.03      0.06        29
           2       0.97      0.90      0.93       123
           3       1.00      0.33      0.50         6
           4       0.72      0.98      0.83       125

    accuracy                           0.81       293
   macro avg       0.60      0.45      0.46       293
weighted avg       0.76      0.81      0.76       293



Looking at the best parameter combination the grid search settled on.

In [25]:
print (model.best_estimator_)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=1, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)
